# SLIMsequence: POS-only fake news detection

We fine-tune XLNet-base on adjectives and adverbs extracted from ReCOVery and Fake and Real News, with caps of 10%, 15%, and 20% of the original token count. Labels: 0 = fake, 1 = real.

## 1. Setup and data loading

In [1]:
# Install dependencies for POS extraction and XLNet classification
!pip install -q transformers accelerate sentencepiece nltk

In [2]:
# Import libraries for data processing and model training
import re
import math
import random
import numpy as np
import pandas as pd
import nltk
import torch
import transformers
import sklearn

from tqdm.auto import tqdm

# Display library versions for reproducibility
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("NLTK:", nltk.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)

from IPython.display import display
import time
import gc

PyTorch: 2.14.1+cu130
Transformers: 5.22.1
NLTK: 3.9.1
NumPy: 2.4.6
Pandas: 2.2.3
Scikit-learn: 1.6.1


In [3]:
# Define the original SLIM repository dataset location
base_url = "https://raw.githubusercontent.com/kappakant/SLIM/main/dataset"

files = {
    "ReCOVery": {
        "train": f"{base_url}/ReCOVery/train_preprocessed.csv",
        "validation": f"{base_url}/ReCOVery/eval_preprocessed.csv",
        "test": f"{base_url}/ReCOVery/test_preprocessed.csv"
    },
    "Fake_And_Real_News": {
        "train": f"{base_url}/fake_and_real_news/train_preprocessed.csv",
        "validation": f"{base_url}/fake_and_real_news/eval_preprocessed.csv",
        "test": f"{base_url}/fake_and_real_news/test_preprocessed.csv"
    }
}

# Load and standardize the original datasets
pos_raw = {}

for dataset_name, splits in files.items():
    pos_raw[dataset_name] = {}

    for split_name, url in splits.items():
        df = pd.read_csv(url)

        # Standardize the article text column across both datasets
        text_column = (
            "text" if dataset_name == "ReCOVery"
            else "merged_info"
        )

        df = df.rename(columns={text_column: "body"})

        # Retain only article text and classification labels
        pos_raw[dataset_name][split_name] = df[
            ["body", "label"]
        ].copy()

        print(dataset_name, split_name, df.shape[0])

ReCOVery train 1453
ReCOVery validation 184
ReCOVery test 392
Fake_And_Real_News train 2297
Fake_And_Real_News validation 1148
Fake_And_Real_News test 1149


In [4]:
# Download resources required for word tokenization and POS tagging
nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger_eng")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

## 2. POS extraction
Tag each article once, retain adjectives and adverbs in their original order, and remove punctuation after applying each cap.

In [5]:
# Generate all POS proportions in a single tagging pass.
def extract_pos_configurations(text, percentages):
    tokens = nltk.word_tokenize(str(text).lower())
    tagged_words = nltk.pos_tag(tokens)

    #Extract adjectives and adverbs (JJ and RB POS tags)
    pos_words = [word for word, tag in tagged_words
                 if tag.startswith("JJ") or tag.startswith("RB")]
    return {n: re.sub(r"[^\w\s]", "", " ".join(pos_words[:math.ceil(len(tokens) * (n / 100))]))
            for n in percentages}

def extract_pos_words(text, n):
    return extract_pos_configurations(text, [n])[n]

In [6]:
# Verify POS extraction using a sample sentence
sample_text = (
    "The extremely dangerous virus spread rapidly "
    "through several large cities."
)

print("Original:", sample_text)

#Show 10%, 15%, 20% POS words from the sample sentence
for n in [10, 15, 20, 100]:
    print(f"POS {n}%:", extract_pos_words(sample_text, n))

Original: The extremely dangerous virus spread rapidly through several large cities.
POS 10%: extremely dangerous
POS 15%: extremely dangerous
POS 20%: extremely dangerous rapidly
POS 100%: extremely dangerous rapidly several large


In [7]:
# Extract POS sequences for each dataset, split and percentage.
pos_percentages = [10, 15, 20]
pos_preprocessed = {n: {} for n in pos_percentages}
for dataset_name, splits in pos_raw.items():
    for n in pos_percentages:
        pos_preprocessed[n][dataset_name] = {}
    for split_name, df in splits.items():
        extracted = {n: [] for n in pos_percentages}
        for text in tqdm(df["body"], desc=f"{dataset_name} - {split_name}"):
            selected = extract_pos_configurations(text, pos_percentages)
            for n in pos_percentages:
                extracted[n].append(selected[n])
        for n in pos_percentages:
            processed_df = df.copy()
            processed_df["body"] = extracted[n]
            pos_preprocessed[n][dataset_name][split_name] = processed_df

ReCOVery - train:   0%|          | 0/1453 [00:00<?, ?it/s]

ReCOVery - validation:   0%|          | 0/184 [00:00<?, ?it/s]

ReCOVery - test:   0%|          | 0/392 [00:00<?, ?it/s]

Fake_And_Real_News - train:   0%|          | 0/2297 [00:00<?, ?it/s]

Fake_And_Real_News - validation:   0%|          | 0/1148 [00:00<?, ?it/s]

Fake_And_Real_News - test:   0%|          | 0/1149 [00:00<?, ?it/s]

## 3. Information reduction and tokenization
Word and token statistics are measured before padding or truncation. XLNet input length is fixed at 512.

In [8]:
# Calculate information reduction statistics for each POS configuration
pos_stats = []

for n, datasets in pos_preprocessed.items():

    for dataset_name, splits in datasets.items():

        for split_name, df in splits.items():

            # Retrieve the corresponding original articles
            original_df = pos_raw[dataset_name][split_name]

            # Calculate word counts before and after POS extraction
            original_counts = original_df["body"].str.split().str.len()
            pos_counts = df["body"].str.split().str.len()

            original_avg = original_counts.mean()
            pos_avg = pos_counts.mean()

            # Calculate the percentage of information removed
            reduction = (1 - pos_avg / original_avg) * 100

            # Record statistics for the current configuration
            pos_stats.append({
                "Dataset": dataset_name,
                "Split": split_name,
                "POS (%)": n,
                "Original Avg Words": original_avg,
                "POS Avg Words": pos_avg,
                "Reduction (%)": reduction,
                "Empty Samples": (pos_counts == 0).sum()
            })

# Display the information reduction statistics
pos_stats_df = pd.DataFrame(pos_stats)

display(pos_stats_df.round(2))

,Dataset,Split,POS (%),Original Avg Words,POS Avg Words,Reduction (%),Empty Samples
0,ReCOVery,train,10,825.60,93.39,88.69,0
1,ReCOVery,validation,10,921.71,103.71,88.75,0
2,ReCOVery,test,10,849.12,96.15,88.68,0
3,Fake_And_Real_News,train,10,817.41,92.40,88.70,32
4,Fake_And_Real_News,validation,10,768.54,86.92,88.69,18
5,Fake_And_Real_News,test,10,812.53,91.66,88.72,10
6,ReCOVery,train,15,825.60,122.87,85.12,0
7,ReCOVery,validation,15,921.71,138.61,84.96,0
8,ReCOVery,test,15,849.12,124.64,85.32,0
9,Fake_And_Real_News,train,15,817.41,123.16,84.93,32


Notice above that there is a 80%-90$ reduction in words after POS extraction

In [9]:
from transformers import XLNetTokenizer

# Using pretrained XLNet tokenizer and truncating to 512
model_name = "xlnet-base-cased"
max_length = 512

tokenizer = XLNetTokenizer.from_pretrained(
    model_name,
    do_lower_case=True
)

# Verify the tokenizer configuration
print("Model:", model_name)
print("Vocabulary size:", tokenizer.vocab_size)
print("Padding side:", tokenizer.padding_side)
print("Lowercase:", tokenizer.do_lower_case)
print("Maximum sequence length:", max_length)

spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

Model: xlnet-base-cased
Vocabulary size: 32000
Padding side: left
Lowercase: True
Maximum sequence length: 512


In [10]:
# Calculate XLNet token counts for the original articles
original_token_counts = {}

for dataset_name, splits in pos_raw.items():

    original_token_counts[dataset_name] = {}

    for split_name, df in splits.items():

        print(f"Original: {dataset_name} - {split_name}")

        original_token_counts[dataset_name][split_name] = [
            len(tokenizer.tokenize(str(text)))
            for text in tqdm(df["body"], total=len(df))
        ]

# Calculate XLNet token statistics for each POS configuration
pos_token_stats = []

for n, datasets in pos_preprocessed.items():

    for dataset_name, splits in datasets.items():

        for split_name, df in splits.items():

            print(f"POS {n}%: {dataset_name} - {split_name}")

            # Retrieve the original article token counts
            original_counts = original_token_counts[
                dataset_name
            ][split_name]

            # Count POS tokens before padding and truncation
            pos_counts = [
                len(tokenizer.tokenize(str(text)))
                for text in tqdm(df["body"], total=len(df))
            ]

            original_avg = np.mean(original_counts)
            pos_avg = np.mean(pos_counts)

            # Calculate the reduction in XLNet tokens
            reduction = (1 - pos_avg / original_avg) * 100

            # Record token statistics and potential truncation
            pos_token_stats.append({
                "Dataset": dataset_name,
                "Split": split_name,
                "POS (%)": n,
                "Original Avg Tokens": original_avg,
                "POS Avg Tokens": pos_avg,
                "Reduction (%)": reduction,
                "Max POS Tokens": max(pos_counts),
                "Samples Over 510 Tokens": sum(
                    count > 510 for count in pos_counts #510 actual tokens + 2 special tokens (<sep> and <cls>)
                )
            })

# Display the token statistics
pos_token_stats_df = pd.DataFrame(pos_token_stats)

display(pos_token_stats_df.round(2))

Original: ReCOVery - train


  0%|          | 0/1453 [00:00<?, ?it/s]

Original: ReCOVery - validation


  0%|          | 0/184 [00:00<?, ?it/s]

Original: ReCOVery - test


  0%|          | 0/392 [00:00<?, ?it/s]

... (39 outputs left out)

  0%|          | 0/1148 [00:00<?, ?it/s]

POS 20%: Fake_And_Real_News - test


  0%|          | 0/1149 [00:00<?, ?it/s]

,Dataset,Split,POS (%),Original Avg Tokens,POS Avg Tokens,Reduction (%),Max POS Tokens,Samples Over 510 Tokens
0,ReCOVery,train,10,1145.55,123.39,89.23,1070,24
1,ReCOVery,validation,10,1297.17,144.77,88.84,3623,4
2,ReCOVery,test,10,1180.81,127.53,89.20,2396,7
3,Fake_And_Real_News,train,10,1149.68,123.82,89.23,2827,45
4,Fake_And_Real_News,validation,10,1082.41,116.08,89.28,1342,14
5,Fake_And_Real_News,test,10,1142.58,122.46,89.28,2258,21
6,ReCOVery,train,15,1145.55,163.36,85.74,1757,39
7,ReCOVery,validation,15,1297.17,194.40,85.01,5293,5
8,ReCOVery,test,15,1180.81,165.22,86.01,2516,15
9,Fake_And_Real_News,train,15,1149.68,164.45,85.70,4239,79


In [11]:
# Convert each POS sequence into input IDs, attention masks and labels.
pos_tokenized = {}
for n, datasets in pos_preprocessed.items():
    pos_tokenized[n] = {}
    for dataset_name, splits in datasets.items():
        pos_tokenized[n][dataset_name] = {}
        for split_name, df in splits.items():
            # Empty strings converted to nan to match authors' implementation.
            texts = df["body"].replace("", np.nan).astype(str).tolist()
            encodings = tokenizer(texts, add_special_tokens=True, truncation=True,
                                  padding="max_length", max_length=max_length,
                                  return_attention_mask=True, return_token_type_ids=False,
                                  return_tensors="pt")
            pos_tokenized[n][dataset_name][split_name] = {
                "input_ids": encodings["input_ids"],
                "attention_mask": encodings["attention_mask"],
                "labels": torch.tensor(df["label"].astype(int).tolist(), dtype=torch.long)
            }

In [12]:
from torch.utils.data import (
    TensorDataset,
    DataLoader,
    RandomSampler,
    SequentialSampler
)

# Match the batch size used in the original implementation
batch_size = 16

pos_dataloaders = {}

# Create DataLoaders for all POS configurations
for n, datasets in pos_tokenized.items():

    pos_dataloaders[n] = {}

    for dataset_name, splits in datasets.items():

        pos_dataloaders[n][dataset_name] = {}

        for split_name, tensors in splits.items():

            # Group input IDs, attention masks, and labels
            dataset = TensorDataset(
                tensors["input_ids"], #token IDs
                tensors["attention_mask"], #Distinguishes actual tokens from padding
                tensors["labels"] #real or fake
            )

            # Shuffle training samples only
            sampler = (
                RandomSampler(dataset)
                if split_name == "train"
                else SequentialSampler(dataset)
            )

            # Create batches for model training and evaluation
            dataloader = DataLoader(
                dataset,
                sampler=sampler,
                batch_size=batch_size
            )

            pos_dataloaders[n][dataset_name][split_name] = dataloader

            print(
                f"POS {n}% | {dataset_name} | {split_name}:",
                len(dataloader), "batches"
            )

POS 10% | ReCOVery | train: 91 batches
POS 10% | ReCOVery | validation: 12 batches
POS 10% | ReCOVery | test: 25 batches
POS 10% | Fake_And_Real_News | train: 144 batches
POS 10% | Fake_And_Real_News | validation: 72 batches
POS 10% | Fake_And_Real_News | test: 72 batches
POS 15% | ReCOVery | train: 91 batches
POS 15% | ReCOVery | validation: 12 batches
POS 15% | ReCOVery | test: 25 batches
POS 15% | Fake_And_Real_News | train: 144 batches
POS 15% | Fake_And_Real_News | validation: 72 batches
POS 15% | Fake_And_Real_News | test: 72 batches
POS 20% | ReCOVery | train: 91 batches
POS 20% | ReCOVery | validation: 12 batches
POS 20% | ReCOVery | test: 25 batches
POS 20% | Fake_And_Real_News | train: 144 batches
POS 20% | Fake_And_Real_News | validation: 72 batches
POS 20% | Fake_And_Real_News | test: 72 batches


## 4. XLNet fine-tuning
Each of the six POS configurations is trained independently from pretrained XLNet-base for eight epochs. We use learning rate 2e-5, batch size 16, fixed padding, full precision, and seed 42. The best model is selected by batch-averaged validation accuracy.

In [13]:
# Load the legacy Hugging Face AdamW optimizer for compatibility with the original implementation.

import requests
import ast
import warnings

from typing import Callable, Iterable, Tuple
from torch import nn
from torch.optim import Optimizer
from transformers.utils.versions import require_version

# Retrieve the legacy Hugging Face AdamW implementation
url = (
    "https://raw.githubusercontent.com/huggingface/"
    "transformers/v4.30.2/src/transformers/optimization.py"
)

response = requests.get(url, timeout=30)
response.raise_for_status()

# Parse the source code and isolate the AdamW class
source_tree = ast.parse(response.text)

adamw_class = next(
    node for node in source_tree.body
    if isinstance(node, ast.ClassDef) and node.name == "AdamW"
)

# Load the legacy optimizer without downgrading Transformers
legacy_module = ast.Module(
    body=[adamw_class],
    type_ignores=[]
)

exec(compile(legacy_module, "<legacy_adamw>", "exec"))

LegacyAdamW = AdamW

print("Legacy Hugging Face AdamW loaded successfully!")

Legacy Hugging Face AdamW loaded successfully!


In [14]:
# Shared hyperparameters for all POS experiments.
from transformers import XLNetForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
seed = 42
learning_rate = 2e-5
epsilon = 1e-8
weight_decay = 0.0
max_grad_norm = 1.0
epochs = 8

def reset_seed():
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [15]:
# Train the model for one complete epoch
def train_one_epoch(model, dataloader, optimizer, scheduler):

    model.train()

    total_loss = 0
    start_time = time.time()

    for batch in tqdm(dataloader, desc="Training"):

        # Transfer the batch to the GPU
        input_ids, attention_mask, labels = [
            tensor.to(device) for tensor in batch
        ]

        # Clear gradients from the previous iteration
        optimizer.zero_grad(set_to_none=True)

        # Forward pass and cross-entropy loss calculation
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        # Backpropagate and clip gradients
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_grad_norm
        )

        # Update model parameters and learning rate
        optimizer.step()
        scheduler.step()

    avg_loss = total_loss / len(dataloader)

    return avg_loss, time.time() - start_time


# Evaluate classification accuracy on the validation set
def validate(model, dataloader):

    model.eval()

    total_accuracy = 0
    start_time = time.time()

    # Disable gradient computation during validation
    with torch.no_grad():

        for batch in tqdm(dataloader, desc="Validation"):

            input_ids, attention_mask, labels = [
                tensor.to(device) for tensor in batch
            ]

            # Generate predictions from classification logits
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            predictions = torch.argmax(outputs.logits, dim=1)

            # Calculate accuracy for the current batch
            batch_accuracy = (
                predictions == labels
            ).float().mean().item()

            total_accuracy += batch_accuracy

    # Match the authors' batch-averaged validation accuracy
    avg_accuracy = total_accuracy / len(dataloader)

    return avg_accuracy, time.time() - start_time

print("Training and validation functions ready!")

Training and validation functions ready!


In [16]:
# Evaluate the validation-selected model on held-out test data.
def evaluate(model, dataloader):
    model.eval()
    labels, predictions, probabilities = [], [], []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Testing"):
            ids, mask, y = [tensor.to(device) for tensor in batch]
            logits = model(input_ids=ids, attention_mask=mask).logits
            labels.extend(y.cpu().tolist())
            predictions.extend(logits.argmax(dim=1).cpu().tolist())
            probabilities.extend(logits.softmax(dim=1)[:, 1].cpu().tolist())
    print(classification_report(labels, predictions, target_names=["Fake", "Real"], digits=4))
    print(confusion_matrix(labels, predictions))
    return {"accuracy": accuracy_score(labels, predictions),
            "macro_f1": f1_score(labels, predictions, average="macro"),
            "auc_paper": roc_auc_score(labels, predictions),
            "auc_probability": roc_auc_score(labels, probabilities)}

In [17]:
# Train a fresh model for each configuration and retain only its best weights in CPU memory.
def run_experiment(dataset_name, n, epochs):
    loaders = pos_dataloaders[n][dataset_name]
    reset_seed()
    model = XLNetForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)
    optimizer = LegacyAdamW(model.parameters(), lr=learning_rate, eps=epsilon,
                            weight_decay=weight_decay, correct_bias=True)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0,
                 num_training_steps=len(loaders["train"]) * epochs)
    reset_seed()
    best_accuracy, best_epoch, best_weights = -float("inf"), 0, None
    history = []
    for epoch in range(1, epochs + 1):
        loss, train_time = train_one_epoch(model, loaders["train"], optimizer, scheduler)
        accuracy, val_time = validate(model, loaders["validation"])
        history.append(dict(epoch=epoch, train_loss=loss, val_accuracy=accuracy,
                            train_time=train_time, val_time=val_time))
        print(f"Epoch {epoch}/{epochs}: loss={loss:.4f}, validation accuracy={accuracy:.4f}")
        if accuracy > best_accuracy:
            best_accuracy, best_epoch = accuracy, epoch
            # Independent CPU copy avoids retaining another model on GPU.
            best_weights = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    del optimizer, scheduler
    model.zero_grad(set_to_none=True)
    model.load_state_dict(best_weights)
    del best_weights
    metrics = evaluate(model, loaders["test"])
    return dict(dataset=dataset_name, POS=n, epochs=epochs, best_epoch=best_epoch,
                best_validation_accuracy=best_accuracy, **metrics), pd.DataFrame(history)

## 5. Train and evaluate all configurations
Each run starts from pretrained weights and uses only the training split for optimization, the validation split for model selection, and the test split for final evaluation.

### 5a. ReCOVery experiments


In [ ]:
# Train the three ReCOVery configurations independently for eight epochs each.
assert torch.cuda.is_available(), "Select a GPU runtime before training."
results = []
training_histories = {}

for dataset_name in ["ReCOVery"]:
    for n in pos_percentages:
        print(f"\n{dataset_name} | POS {n}%")
        gc.collect()
        torch.cuda.empty_cache()
        result, history = run_experiment(dataset_name, n, epochs)
        results.append(result)
        training_histories[(dataset_name, n)] = history
        gc.collect()
        torch.cuda.empty_cache()
        display(pd.DataFrame(results).round(4))


ReCOVery | POS 10%


config.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  467MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

[transformers] XLNetForSequenceClassification LOAD REPORT from: xlnet-base-cased
Key                             | Status     | 
--------------------------------+------------+-
lm_loss.weight                  | UNEXPECTED | 
lm_loss.bias                    | UNEXPECTED | 
sequence_summary.summary.bias   | MISSING    | 
sequence_summary.summary.weight | MISSING    | 
logits_proj.weight              | MISSING    | 
logits_proj.bias                | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
<legacy_adamw>:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning


... (85 outputs left out)

Epoch 8/8: loss=0.0725, validation accuracy=0.8646


Testing:   0%|          | 0/25 [00:00<?, ?it/s]

              precision    recall  f1-score   support

        Fake     0.7699    0.7632    0.7665       114
        Real     0.9032    0.9065    0.9048       278

    accuracy                         0.8648       392
   macro avg     0.8366    0.8348    0.8357       392
weighted avg     0.8645    0.8648    0.8646       392

[[ 87  27]
 [ 26 252]]


,dataset,POS,epochs,best_epoch,best_validation_accuracy,accuracy,macro_f1,auc_paper,auc_probability
0,ReCOVery,10,8,5,0.8854,0.8597,0.8277,0.8235,0.9192
1,ReCOVery,15,8,3,0.8594,0.8571,0.8341,0.8501,0.9086
2,ReCOVery,20,8,3,0.8646,0.8648,0.8357,0.8348,0.9109


### 5b. Continue with Fake and Real News
 ReCOVery metrics are transcribed from the actual saved outputs (four-decimal precision).

In [19]:
# Restore the actual ReCOVery results from the completed run above (printed to four decimals).
results = [{'dataset': 'ReCOVery', 'POS': 10, 'epochs': 8, 'best_epoch': 5, 'best_validation_accuracy': 0.8854, 'accuracy': 0.8597, 'macro_f1': 0.8277, 'auc_paper': 0.8235, 'auc_probability': 0.9192}, {'dataset': 'ReCOVery', 'POS': 15, 'epochs': 8, 'best_epoch': 3, 'best_validation_accuracy': 0.8594, 'accuracy': 0.8571, 'macro_f1': 0.8341, 'auc_paper': 0.8501, 'auc_probability': 0.9086}, {'dataset': 'ReCOVery', 'POS': 20, 'epochs': 8, 'best_epoch': 3, 'best_validation_accuracy': 0.8646, 'accuracy': 0.8648, 'macro_f1': 0.8357, 'auc_paper': 0.8348, 'auc_probability': 0.9109}]
# Restore the displayed epoch logs; timings were not printed.
training_histories = {("ReCOVery", n): pd.DataFrame(history) for n, history in {10: [{'epoch': 1, 'train_loss': 0.64, 'val_accuracy': 0.7083}, {'epoch': 2, 'train_loss': 0.4696, 'val_accuracy': 0.7917}, {'epoch': 3, 'train_loss': 0.3827, 'val_accuracy': 0.8594}, {'epoch': 4, 'train_loss': 0.257, 'val_accuracy': 0.875}, {'epoch': 5, 'train_loss': 0.1882, 'val_accuracy': 0.8854}, {'epoch': 6, 'train_loss': 0.1414, 'val_accuracy': 0.8854}, {'epoch': 7, 'train_loss': 0.0956, 'val_accuracy': 0.8802}, {'epoch': 8, 'train_loss': 0.0696, 'val_accuracy': 0.875}], 15: [{'epoch': 1, 'train_loss': 0.5629, 'val_accuracy': 0.8177}, {'epoch': 2, 'train_loss': 0.4183, 'val_accuracy': 0.7396}, {'epoch': 3, 'train_loss': 0.3568, 'val_accuracy': 0.8594}, {'epoch': 4, 'train_loss': 0.2307, 'val_accuracy': 0.8542}, {'epoch': 5, 'train_loss': 0.1674, 'val_accuracy': 0.8542}, {'epoch': 6, 'train_loss': 0.1312, 'val_accuracy': 0.8385}, {'epoch': 7, 'train_loss': 0.0912, 'val_accuracy': 0.8333}, {'epoch': 8, 'train_loss': 0.0659, 'val_accuracy': 0.8542}], 20: [{'epoch': 1, 'train_loss': 0.5721, 'val_accuracy': 0.6875}, {'epoch': 2, 'train_loss': 0.4242, 'val_accuracy': 0.7448}, {'epoch': 3, 'train_loss': 0.3204, 'val_accuracy': 0.8646}, {'epoch': 4, 'train_loss': 0.2503, 'val_accuracy': 0.8646}, {'epoch': 5, 'train_loss': 0.171, 'val_accuracy': 0.849}, {'epoch': 6, 'train_loss': 0.1341, 'val_accuracy': 0.8438}, {'epoch': 7, 'train_loss': 0.1002, 'val_accuracy': 0.8385}, {'epoch': 8, 'train_loss': 0.0725, 'val_accuracy': 0.8646}]}.items()}
display(pd.DataFrame(results))

,dataset,POS,epochs,best_epoch,best_validation_accuracy,accuracy,macro_f1,auc_paper,auc_probability
0,ReCOVery,10,8,5,0.8854,0.8597,0.8277,0.8235,0.9192
1,ReCOVery,15,8,3,0.8594,0.8571,0.8341,0.8501,0.9086
2,ReCOVery,20,8,3,0.8646,0.8648,0.8357,0.8348,0.9109


In [ ]:
# Train only the three unfinished Fake and Real News configurations.
assert torch.cuda.is_available(), "Select a GPU runtime before training."
for n in pos_percentages:
    if any(r["dataset"] == "Fake_And_Real_News" and r["POS"] == n for r in results):
        continue
    print(f"\nFake_And_Real_News | POS {n}%")
    gc.collect()
    torch.cuda.empty_cache()
    result, history = run_experiment("Fake_And_Real_News", n, epochs)
    results.append(result)
    training_histories[("Fake_And_Real_News", n)] = history
    gc.collect()
    torch.cuda.empty_cache()
    display(pd.DataFrame(results).round(4))


Fake_And_Real_News | POS 10%


config.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  467MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

[transformers] XLNetForSequenceClassification LOAD REPORT from: xlnet-base-cased
Key                             | Status     | 
--------------------------------+------------+-
lm_loss.bias                    | UNEXPECTED | 
lm_loss.weight                  | UNEXPECTED | 
logits_proj.weight              | MISSING    | 
sequence_summary.summary.weight | MISSING    | 
logits_proj.bias                | MISSING    | 
sequence_summary.summary.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


... (69 outputs left out)

Training:   0%|          | 0/144 [00:00<?, ?it/s]

Validation:   0%|          | 0/72 [00:00<?, ?it/s]

Epoch 3/8: loss=0.2328, validation accuracy=0.8299


Training:   0%|          | 0/144 [00:00<?, ?it/s]

In [19]:
# Train only Fake and Real News — POS 20%
#Retraining only 20% POS because the previous run was interrupted by a GPU limit
assert torch.cuda.is_available(), "Select a GPU runtime."

n = 20
dataset_name = "Fake_And_Real_News"

gc.collect()
torch.cuda.empty_cache()

result, history = run_experiment(dataset_name, n, epochs)

results.append(result)
training_histories[(dataset_name, n)] = history

display(pd.DataFrame(results).round(4))

config.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  467MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

[transformers] XLNetForSequenceClassification LOAD REPORT from: xlnet-base-cased
Key                             | Status     | 
--------------------------------+------------+-
lm_loss.bias                    | UNEXPECTED | 
lm_loss.weight                  | UNEXPECTED | 
sequence_summary.summary.bias   | MISSING    | 
logits_proj.bias                | MISSING    | 
logits_proj.weight              | MISSING    | 
sequence_summary.summary.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
<legacy_adamw>:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning


Training:   0%|          | 0/144 [00:00<?, ?it/s]

... (24 outputs left out)

Epoch 8/8: loss=0.0396, validation accuracy=0.8903


Testing:   0%|          | 0/72 [00:00<?, ?it/s]

              precision    recall  f1-score   support

        Fake     0.8878    0.8953    0.8915       592
        Real     0.8877    0.8797    0.8837       557

    accuracy                         0.8877      1149
   macro avg     0.8877    0.8875    0.8876      1149
weighted avg     0.8877    0.8877    0.8877      1149

[[530  62]
 [ 67 490]]


NameError: name 'results' is not defined

In [ ]:
# Compare held-out test metrics across all trained configurations.
results_df = pd.DataFrame(results)
display(results_df.round(4))

## Limitations
This is a single-seed reproduction with a reduced learning rate (2e-5 rather than 5e-5) and 8 epochs rather than the original implementation’s 15. Hard-label AUC follows the authors’ implementation; probability-based ROC AUC is also reported.